# Canvas API Interface

This notebook provides a Python interface to the Canvas LMS API for managing course submissions and grades.

## Setup

First, store your Canvas API token. For security, we'll load it from a `.env` file rather than hardcoding it.

In [11]:
# Create a .env file if it doesn't exist
# You only need to run this once, then add your token to the file

import os
from pathlib import Path

env_path = Path('../.env')
if not env_path.exists():
    env_path.write_text('CANVAS_TOKEN=your_token_here\n')
    print(f'Created {env_path.absolute()}')
    print('Please edit the .env file and add your Canvas API token')
else:
    print(f'.env file already exists at {env_path.absolute()}')

.env file already exists at c:\Users\jonat\My Drive\Hunter\eco331\code\..\.env


In [2]:
# Load the token from .env file

import os
from pathlib import Path

def load_env(filepath='../.env'):
    """Load environment variables from a .env file."""
    env_path = Path(filepath)
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                key, value = line.split('=', 1)
                os.environ[key.strip()] = value.strip()
        print('Loaded .env file')
    else:
        print(f'Warning: {filepath} not found')

load_env()

CANVAS_TOKEN = os.environ.get('CANVAS_TOKEN')
if CANVAS_TOKEN and CANVAS_TOKEN != 'your_token_here':
    print('Canvas token loaded successfully')
else:
    print('Please add your Canvas token to the .env file')

Loaded .env file
Canvas token loaded successfully


In [3]:
# Canvas API configuration

import requests

CANVAS_BASE_URL = 'https://canvas.instructure.com/api/v1'

def canvas_request(endpoint, method='GET', params=None, data=None):
    """Make a request to the Canvas API."""
    url = f'{CANVAS_BASE_URL}/{endpoint}'
    headers = {
        'Authorization': f'Bearer {CANVAS_TOKEN}'
    }
    
    response = requests.request(
        method=method,
        url=url,
        headers=headers,
        params=params,
        json=data
    )
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f'Error {response.status_code}: {response.text}')
        return None

## Test Connection

Verify that the API connection works by fetching your user profile.

In [4]:
# Test: Get current user info

user = canvas_request('users/self')
if user:
    print(f"Connected as: {user['name']}")
    print(f"User ID: {user['id']}")

Connected as: Jonathan Conning
User ID: 15131563


## List Courses

Fetch your active courses to find the course ID for ECO 331.

In [5]:
# List active courses

courses = canvas_request('courses', params={'enrollment_state': 'active'})
if courses:
    print('Your active courses:\n')
    for course in courses:
        course_id = course.get('id')
        name = course.get('name', 'Unnamed')
        code = course.get('course_code', '')
        print(f"  ID: {course_id}")
        print(f"  Name: {name}")
        print(f"  Code: {code}")
        print()

Your active courses:

  ID: 11204793
  Name: Eco 331: Economic History, SP25
  Code: Eco 331

  ID: 14011875
  Name: Eco 331: Economic History, Spring 2026
  Code: Eco 331:

  ID: 1345003
  Name: Your Guided Course Template
  Code: CANVAS-NA



## List Assignments

Once you have a course ID, list its assignments.

In [6]:
# Set your course ID here (from the list above)
COURSE_ID = 14011875  # Replace with your ECO 331 course ID

if COURSE_ID:
    assignments = canvas_request(f'courses/{COURSE_ID}/assignments')
    if assignments:
        print(f'Assignments for course {COURSE_ID}:\n')
        for a in assignments:
            print(f"  ID: {a['id']}")
            print(f"  Name: {a['name']}")
            print(f"  Due: {a.get('due_at', 'No due date')}")
            print()
else:
    print('Please set COURSE_ID above')

Assignments for course 14011875:

  ID: 61516327
  Name: Introduce Yourself
  Due: None

  ID: 61516331
  Name: (Friday 1/31) Intro: Q & Comments
  Due: 2025-02-08T06:30:00Z

  ID: 61516333
  Name: (Friday 2/7)  Neolithic Revolution: Q & Comments (sorry posted late!)
  Due: 2025-02-08T06:30:00Z

  ID: 61516351
  Name: (Tue 2/11) Geography: Q & Comments (abbreviated)
  Due: 2025-02-11T16:30:00Z

  ID: 61516329
  Name: (Fri 2/14) Malthusian Epoch:  Q & comments
  Due: 2025-02-14T16:30:00Z

  ID: 61516335
  Name: (Mon 2/21): OG Institutions Q & comments
  Due: 2025-02-21T16:30:00Z

  ID: 61516353
  Name: (Tue 3/11) Reading KR 7
  Due: 2025-03-11T15:30:00Z

  ID: 61516345
  Name: (Sun 2/26) Short essay
  Due: 2023-02-27T06:59:00Z

  ID: 61516349
  Name: (Thurs 3/2) Culture +
  Due: 2023-03-03T05:00:00Z

  ID: 61516347
  Name: (Thur 03/14) Q & C on Medieval economy 
  Due: 2023-02-28T06:59:00Z



## Get Submissions

Fetch student submissions for a specific assignment.

In [7]:
# Set your assignment ID here
ASSIGNMENT_ID = 14011875  # Replace with the assignment ID you want to grade

if COURSE_ID and ASSIGNMENT_ID:
    submissions = canvas_request(
        f'courses/{COURSE_ID}/assignments/{ASSIGNMENT_ID}/submissions',
        params={'include[]': ['user', 'submission_comments']}
    )
    if submissions:
        print(f'Submissions for assignment {ASSIGNMENT_ID}:\n')
        for sub in submissions:
            user_name = sub.get('user', {}).get('name', 'Unknown')
            submitted = sub.get('submitted_at', 'Not submitted')
            score = sub.get('score', 'Not graded')
            body = sub.get('body', '')[:100] if sub.get('body') else 'No text submission'
            
            print(f"  Student: {user_name}")
            print(f"  Submitted: {submitted}")
            print(f"  Score: {score}")
            print(f"  Preview: {body}...")
            print()
else:
    print('Please set COURSE_ID and ASSIGNMENT_ID above')

Error 404: {"errors":[{"message":"The specified resource does not exist."}]}


## Download File Submissions

When students submit file attachments (PDFs, Word docs, etc.), you can download them programmatically. This is useful for:
- Batch downloading all submissions for offline grading
- Processing submissions with external tools
- Creating backups of student work

The key is that you specify which assignment to download from via the API endpoint.

In [ ]:
# Download file submissions for a specific assignment
# Make sure COURSE_ID is set above, then set the assignment ID here

DOWNLOAD_ASSIGNMENT_ID = None  # Replace with the assignment ID that has file uploads

# Create a folder to store downloads
download_folder = Path(f'./submissions_{DOWNLOAD_ASSIGNMENT_ID}')

if COURSE_ID and DOWNLOAD_ASSIGNMENT_ID:
    download_folder.mkdir(exist_ok=True)
    
    # Get submissions with user info
    submissions = canvas_request(
        f'courses/{COURSE_ID}/assignments/{DOWNLOAD_ASSIGNMENT_ID}/submissions',
        params={'include[]': ['user']}
    )
    
    if submissions:
        file_count = 0
        for sub in submissions:
            student_name = sub.get('user', {}).get('name', 'Unknown')
            # Clean student name for use in filename
            safe_name = "".join(c for c in student_name if c.isalnum() or c in (' ', '-', '_')).strip()
            
            attachments = sub.get('attachments', [])
            if not attachments:
                continue
                
            for attachment in attachments:
                file_url = attachment['url']
                filename = attachment['filename']
                
                # Save with student name prefix to keep organized
                save_as = download_folder / f"{safe_name}_{filename}"
                
                print(f"Downloading: {save_as.name}")
                response = requests.get(
                    file_url,
                    headers={'Authorization': f'Bearer {CANVAS_TOKEN}'}
                )
                
                if response.status_code == 200:
                    save_as.write_bytes(response.content)
                    file_count += 1
                else:
                    print(f"  Error downloading: {response.status_code}")
        
        print(f"\nDownloaded {file_count} files to {download_folder.absolute()}")
    else:
        print("No submissions found")
else:
    print('Please set COURSE_ID and DOWNLOAD_ASSIGNMENT_ID above')

## Upload Markdown to Canvas Pages

You can write content in markdown locally (e.g., in your Obsidian vault) and push it to Canvas as a page. The markdown is converted to HTML before uploading.

**Requirements:**
- `pip install markdown` (for converting markdown to HTML)

**Notes:**
- Images must use remote URLs (Google Drive, Canvas Files, etc.)
- Obsidian-style `[[wikilinks]]` won't work—use standard markdown links
- The page URL slug is derived from the title (e.g., "Week 3 Notes" → `week-3-notes`)

In [8]:
import markdown

def upload_markdown_to_canvas(course_id, title, md_content, published=False):
    """
    Convert markdown to HTML and upload as a Canvas page.
    
    Args:
        course_id: Canvas course ID
        title: Page title (also determines the URL slug)
        md_content: Markdown content as a string
        published: Whether to publish immediately (default False = draft)
    
    Returns:
        The created/updated page data, or None if failed
    """
    # Convert markdown to HTML
    # Enable extensions for tables, fenced code blocks, etc.
    html_content = markdown.markdown(
        md_content,
        extensions=['tables', 'fenced_code', 'toc']
    )
    
    # Create URL slug from title
    url_slug = title.lower().replace(' ', '-')
    url_slug = ''.join(c for c in url_slug if c.isalnum() or c == '-')
    
    # Try to update existing page, or create new one
    result = canvas_request(
        f'courses/{course_id}/pages/{url_slug}',
        method='PUT',
        data={
            'wiki_page': {
                'title': title,
                'body': html_content,
                'published': published
            }
        }
    )
    
    if result:
        status = "published" if published else "draft"
        print(f"Page '{title}' uploaded successfully ({status})")
        print(f"URL: https://canvas.instructure.com/courses/{course_id}/pages/{url_slug}")
    
    return result

In [ ]:
# First, install html2text if needed
import subprocess
import sys

try:
    import html2text
except ImportError:
    print("Installing html2text...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "html2text", "-q"])
    import html2text

# Download all Canvas pages as markdown files
def download_canvas_pages_to_markdown(course_id, output_dir='canvas'):
    """
    Download all pages from a Canvas course and save as markdown files.
    
    Args:
        course_id: Canvas course ID
        output_dir: Directory to save markdown files (default: canvas)
    """
    output_path = Path(output_dir).resolve()  # Use absolute path from project root
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Get list of pages (doesn't include body by default)
    pages_list = canvas_request(f'courses/{course_id}/pages')
    
    if not pages_list:
        print("No pages found")
        return
    
    # Initialize HTML to Markdown converter
    h = html2text.HTML2Text()
    h.ignore_links = False
    h.body_width = 0  # Don't wrap lines
    
    print(f"Downloading {len(pages_list)} pages to {output_path}\n")
    
    downloaded = 0
    for page_summary in pages_list:
        title = page_summary.get('title', 'Untitled')
        page_url = page_summary.get('url', '')
        
        # Fetch the full page content (includes body)
        page = canvas_request(f'courses/{course_id}/pages/{page_url}')
        
        if not page:
            print(f"  Error: Could not fetch {title}")
            continue
        
        html_body = page.get('body', '')
        
        if not html_body:
            print(f"  Skipped: {title} (no content)")
            continue
        
        # Convert HTML to markdown
        md_content = h.handle(html_body)
        
        # Create filename from title
        filename = title.lower().replace(' ', '_').replace(':', '').replace('?', '')
        # Remove any remaining special characters
        filename = ''.join(c for c in filename if c.isalnum() or c in ('_', '-'))
        filename = f"{filename}.md"
        
        filepath = output_path / filename
        
        # Write markdown file
        filepath.write_text(md_content, encoding='utf-8')
        
        status = "published" if page.get('published') else "draft"
        print(f"  ✓ {title}")
        print(f"     → {filename} ({status})")
        downloaded += 1
    
    print(f"\nDownloaded {downloaded} pages to {output_path}")

# Run the download
if COURSE_ID:
    download_canvas_pages_to_markdown(COURSE_ID)
else:
    print("Please set COURSE_ID above")

## Download Canvas Pages as Markdown

Download all existing Canvas pages and save them as markdown files to your `./canvas` folder. This lets you edit them locally and version control them.

In [21]:
# First, install html2text and frontmatter if needed
import subprocess
import sys
import html2text

try:
    import frontmatter
except ImportError:
    print("Installing python-frontmatter...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-frontmatter", "-q"])
    import frontmatter

# Download all Canvas pages as markdown files with YAML frontmatter
def download_canvas_pages_to_markdown(course_id, output_dir='canvas'):
    """
    Download all pages from a Canvas course and save as markdown files with YAML frontmatter.
    
    YAML frontmatter includes:
    - published: true/false (preserves publication status)
    - title: Page title (for reference when uploading)
    
    Args:
        course_id: Canvas course ID
        output_dir: Directory to save markdown files (default: canvas)
    """
    output_path = Path(output_dir).resolve()  # Use absolute path from project root
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Get list of pages (doesn't include body by default)
    pages_list = canvas_request(f'courses/{course_id}/pages')
    
    if not pages_list:
        print("No pages found")
        return
    
    # Initialize HTML to Markdown converter
    h = html2text.HTML2Text()
    h.ignore_links = False
    h.body_width = 0  # Don't wrap lines
    
    print(f"Downloading {len(pages_list)} pages to {output_path}\n")
    
    downloaded = 0
    for page_summary in pages_list:
        title = page_summary.get('title', 'Untitled')
        page_url = page_summary.get('url', '')
        
        # Fetch the full page content (includes body)
        page = canvas_request(f'courses/{course_id}/pages/{page_url}')
        
        if not page:
            print(f"  Error: Could not fetch {title}")
            continue
        
        html_body = page.get('body', '')
        
        if not html_body:
            print(f"  Skipped: {title} (no content)")
            continue
        
        # Convert HTML to markdown
        md_content = h.handle(html_body)
        
        # Create filename from title
        filename = title.lower().replace(' ', '_').replace(':', '').replace('?', '')
        # Remove any remaining special characters
        filename = ''.join(c for c in filename if c.isalnum() or c in ('_', '-'))
        filename = f"{filename}.md"
        
        filepath = output_path / filename
        
        # Create YAML frontmatter with published status and title
        post = frontmatter.Post(md_content)
        post.metadata['published'] = page.get('published', False)
        post.metadata['title'] = title
        
        # Write markdown file with frontmatter
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(frontmatter.dumps(post))
        
        status = "published" if page.get('published') else "draft"
        print(f"  ✓ {title}")
        print(f"     → {filename} ({status})")
        downloaded += 1
    
    print(f"\nDownloaded {downloaded} pages to {output_path}")

# Run the download
if COURSE_ID:
    download_canvas_pages_to_markdown(COURSE_ID)
else:
    print("Please set COURSE_ID above")


  ✓ Course Outline And Reading Schedule
     → course_outline_and_reading_schedule.md (published)
  ✓ Course Slides
     → course_slides.md (draft)
  ✓ Course Syllabus
     → course_syllabus.md (published)
  ✓ Final Exam Questions
     → final_exam_questions.md (draft)
  ✓ Google Scholar And Zotero
     → google_scholar_and_zotero.md (draft)
  ✓ Instructor And Contact Info
     → instructor_and_contact_info.md (draft)
  ✓ Midterm Materials
     → midterm_materials.md (draft)
  ✓ Paper Presentations
     → paper_presentations.md (draft)
  ✓ Week 4 Malthusian Economics
     → week_4_malthusian_economics.md (draft)
  ✓ Paper Presentation Schedule
     → paper_presentation_schedule.md (published)

Downloaded 10 pages to C:\Users\jonat\My Drive\Hunter\eco331\code\canvas


## Upload Modified Markdown Files to Canvas

Upload all markdown files from your canvas folder back to Canvas pages. This updates existing pages or creates new ones if they don't exist.

In [22]:
# Install python-frontmatter if needed
try:
    import frontmatter
except ImportError:
    print("Installing python-frontmatter...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-frontmatter", "-q"])
    import frontmatter

# Upload all markdown files from canvas folder to Canvas pages (with YAML frontmatter support)

def upload_all_markdown_files(course_id, canvas_folder='canvas', default_published=False):
    """
    Upload all markdown files from a folder to Canvas pages.
    
    Supports YAML frontmatter in markdown files:
    ---
    published: true
    title: Custom Title (optional)
    ---
    
    Args:
        course_id: Canvas course ID
        canvas_folder: Folder containing markdown files (default: canvas)
        default_published: Default published status if not specified in file (default: False)
    """
    canvas_path = Path(canvas_folder)
    
    if not canvas_path.exists():
        print(f"Error: {canvas_folder} folder not found")
        return
    
    # Find all markdown files
    md_files = list(canvas_path.glob('*.md'))
    
    if not md_files:
        print(f"No markdown files found in {canvas_folder}")
        return
    
    print(f"Uploading {len(md_files)} markdown files to Canvas\n")
    
    uploaded = 0
    for md_file in md_files:
        # Parse the file with frontmatter
        post = frontmatter.load(str(md_file))
        
        # Extract metadata
        metadata = post.metadata
        md_content = post.content
        
        # Get title from frontmatter or filename
        title = metadata.get('title') or md_file.stem.replace('_', ' ').title()
        
        # Get published status from frontmatter or use default
        published = metadata.get('published', default_published)
        
        # Upload to Canvas
        result = upload_markdown_to_canvas(course_id, title, md_content, published=published)
        
        if result:
            status = "✓ published" if published else "✓ draft"
            print(f"  {status}: {md_file.name}")
            uploaded += 1
        else:
            print(f"  ✗ Failed to upload {md_file.name}\n")
    
    print(f"\nSuccessfully uploaded {uploaded}/{len(md_files)} pages")

# Run the upload
if COURSE_ID:
    upload_all_markdown_files(COURSE_ID, canvas_folder='canvas')
else:
    print("Please set COURSE_ID above")

Uploading 10 markdown files to Canvas

Page 'Course Outline And Reading Schedule' uploaded successfully (published)
URL: https://canvas.instructure.com/courses/14011875/pages/course-outline-and-reading-schedule
  ✓ published: course_outline_and_reading_schedule.md
Page 'Course Slides' uploaded successfully (draft)
URL: https://canvas.instructure.com/courses/14011875/pages/course-slides
  ✓ draft: course_slides.md
Page 'Course Syllabus' uploaded successfully (published)
URL: https://canvas.instructure.com/courses/14011875/pages/course-syllabus
  ✓ published: course_syllabus.md
Page 'Final Exam Questions' uploaded successfully (draft)
URL: https://canvas.instructure.com/courses/14011875/pages/final-exam-questions
  ✓ draft: final_exam_questions.md
Page 'Google Scholar And Zotero' uploaded successfully (draft)
URL: https://canvas.instructure.com/courses/14011875/pages/google-scholar-and-zotero
  ✓ draft: google_scholar_and_zotero.md
Page 'Instructor And Contact Info' uploaded successfully